In [ ]:
from google.colab import drive

# Google Drive を /content/drive にマウント
drive.mount('/content/drive')

%cd /content/drive/MyDrive/research/proj-spectrum_denoising_1d/spectrum_denoising_1d/examples/
!pip install pytorch_lightning

In this notebook, we use the trained noise model to guide the training of a VAE for denoising.

This version (v2) uses **positional-encoding augmented** data with shape
`(n_samples, 1, channel, 1+d_model)`.

- The VAE receives PE data as input but only processes the scalar channel
  internally, producing output of shape `(n_samples, 1, channel)`.
- The estimated noise `input[:,:,:,0] - output` is concatenated with the PE
  features from the input and fed to the trained noise model for likelihood
  evaluation (replacing the standard reconstruction loss).


In [ ]:
import sys
import os

import torch
import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

sys.path.append("../")
from noise_model.PixelCNN import PixelCNN
from HDN.models.lvae import LadderVAE
from utils.dataloaders import create_dn_loader

Load noisy measurements with positional encoding.
Shape: `[Number, 1, Width, 1+d_model]`

In [ ]:
at_particle_location = f"./../../data/for_1d_denoising/noisy_1d_pe.npy"
at_particle = np.load(at_particle_location)
print("at_particle.shape:", at_particle.shape)
# Expected: (n_samples, 1, channel, 1+d_model) e.g. (100, 1, 200, 5)

Load trained noise model (PE version) and disable gradients.

In [ ]:
noise_model_location = f"../nm_checkpoint_pe/final_params.ckpt"
noise_model = PixelCNN.load_from_checkpoint(noise_model_location).eval()

for param in noise_model.parameters():
    param.requires_grad = False

Create data loaders and get the shape, mean and standard deviation of the noisy images.

For 4D PE data, `Dataset.getparams()` computes mean/std from only the scalar
channel (feature index 0). `getimgwidth()` returns `shape[-1]` which for 4D
data is `1+d_model`. We override `img_width` below to use the actual spatial
width.

In [ ]:
dn_train_loader, dn_val_loader, img_width, data_mean, data_std = create_dn_loader(
    at_particle, batch_size=32, split=0.9
)

# For PE data, img_width from getimgwidth() returns 1+d_model (last dim).
# We need the actual spatial width (channel dim).
img_width = at_particle.shape[-2]
print(f"img_width (channel dim) = {img_width}")
print(f"data_mean = {data_mean}, data_std = {data_std}")

Set denoiser checkpoint directory.

In [ ]:
dn_checkpoint_path = f"../dn_checkpoint_pe"

Initialise trainer and denoiser VAE.

The `LadderVAE` already supports 4D PE input:
- `_extract_signal_channel` extracts `[B, 1, W]` from `[B, 1, W, 1+d_model]`
- `forward` constructs `estimated_noise_pe` by combining predicted noise with
  input PE features and passes it to the noise model for likelihood evaluation.


In [ ]:
use_cuda = torch.cuda.is_available()
trainer = pl.Trainer(
    default_root_dir=dn_checkpoint_path,
    accelerator="gpu" if use_cuda else "cpu",
    devices=1,
    max_epochs=100,
    logger=TensorBoardLogger(dn_checkpoint_path),
    log_every_n_steps=len(dn_train_loader),
    callbacks=[LearningRateMonitor(logging_interval="epoch"),
               EarlyStopping(monitor='val/elbo', patience=50)],
)

num_latents = 8
z_dims = [32] * num_latents
vae = LadderVAE(
    z_dims=z_dims,
    noise_model=noise_model,
    img_width=img_width,
    dropout=0.0,
    data_mean=data_mean,
    data_std=data_std,
)

Train and save final parameters.

In [ ]:
trainer.fit(vae, dn_train_loader, dn_val_loader)
trainer.save_checkpoint(os.path.join(dn_checkpoint_path, "final_params.ckpt"))